In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F

X = torch.rand(3,20)


class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20,256)
        self.out = nn.Linear(256,10)

    def forward(self, X):
        return self.out(F.relu(self.hidden(X)))
    


NN = MLP()
NN(X)

tensor([[-0.1276, -0.1593, -0.0184, -0.0178, -0.0825,  0.0669, -0.0888, -0.0035,
         -0.1536,  0.0722],
        [-0.0853, -0.2060,  0.0230, -0.1592, -0.1433, -0.0088, -0.0679,  0.0252,
         -0.1615,  0.1040],
        [-0.0454, -0.2620,  0.0386, -0.0693, -0.0968,  0.0156,  0.0194,  0.0171,
         -0.1565,  0.1467]], grad_fn=<AddmmBackward0>)

# 块
块代表一个可以接收输入、进行计算并输出结果的独立组件。
pytorch中，万物皆 Module（块），块=nn.Module

## 层
最基础的块，比如nn.Linear，nn.Conv2d。继承自Module的子类

## Sequential
拼接容器，定义串联/单线进程。当构建的模型非常简单，数据只是从一层流向下一层，没有任何分支/复杂的逻辑时，可以使用 Sequential （最方便）。
Sequential甚至不用实现forward函数，同样是继承自Module子类

## Class
必须继承自 nn.Module，从而创造一个自定义的块。
__init__函数中构造/定义所需的层、块，forward实现向前传播的逻辑。

super().__init__()即去执行父类（nn.Module）的 __init__ 初始化函数。

In [19]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for index, block in enumerate(args):
            self._modules[str(index)] = block


    def forward(self, X):
        for block in self._modules.values():
            X = block(X)
        return X
    
NN = MySequential(nn.Linear(20,100), nn.ReLU(), nn.Linear(100,2))
NN(X)

tensor([[ 0.3097, -0.1843],
        [ 0.0916, -0.1710],
        [ 0.1721, -0.1605]], grad_fn=<AddmmBackward0>)

### nn.ReLU() 与 F.relu
nn.Sequential 要求串上去的每一个组件必须都是继承自 nn.Module 的对象，而 F.relu 只是一个普通的函数。
nn.ReLU()是一个继承自Module的块，而F.relu只是一个内置函数
一般在Sequential里面使用（只能）ReLU()，而在自定义块中更建议使用relu：便捷

### args
*args 会打包出一个 args 的元组。enumerate（Python常用的内置函数），意思是“枚举”或“带有索引的遍历”。

### self._modules
self._modules 是 nn.Module 内部自带的有序词典，self._modules[str(idx)] = block 会让pytorch标记/追踪block，这样才会把里面所有子块的权重、偏置提取出来，参与反向传播计算梯度，在保存模型时把它们一并存盘，自动转移CPU/GPU